In [ ]:

import requests, csv, io, tempfile, os
from typing import Optional
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import plotly.io as pio
pio.renderers.default = "notebook_connected"


BASE = "https://transparencia.sns.gov.pt/api/explore/v2.1"
CATALOG_URL = f"{BASE}/catalog/datasets"


Plotly version: 6.5.2
Default renderer: notebook_connected


In [17]:
def list_all_dataset_ids(limit=100, max_rows=100_000, where=None, verbose=True):
    """
    Prints and returns all dataset_id entries from the SNS Transparência catalog.
    - limit:    MUST be <= 100 (Explore API v2.1)
    - max_rows: safety cap
    - where:    optional ODSQL full-text filter on the catalog (e.g., 'cirurgia')
    - verbose:  if True, prints each dataset_id
    """
    if limit > 100:
        raise ValueError("Explore API v2.1 caps 'limit' at 100 for /catalog/datasets. Use pagination.")

    ids = []
    offset = 0
    while True:
        params = {"limit": limit, "offset": offset}
        if where:
            params["where"] = where  # full-text filter if you want to narrow down
        r = requests.get(CATALOG_URL, params=params, timeout=30)
        r.raise_for_status()
        payload = r.json()

        # Different domains may use 'datasets' or 'results'
        rows = payload.get("datasets") or payload.get("results") or []
        if not rows:
            break

        for row in rows:
            # Some instances wrap the object under 'dataset'
            if isinstance(row, dict) and "dataset" in row and isinstance(row["dataset"], dict):
                row = row["dataset"]
            dsid = row.get("dataset_id") if isinstance(row, dict) else None
            if dsid:
                ids.append(dsid)
                if verbose:
                    print(dsid)

        offset += limit
        if len(ids) >= max_rows:
            print(f"Stopped at safety cap max_rows={max_rows}.")
            break

    return pd.DataFrame({"dataset_id": ids})

datasets_df = list_all_dataset_ids(limit=100, max_rows=200_000, verbose=True)


distribuicao-da-reserva-estrategica-de-medicamentos
caracterizacao-das-valencias-de-urgencia
fraturas-da-anca-cirurgias-nas-primeiras-48h
stock-da-reserva-estrategica-de-medicamentos-existentes-a-nivel-central
custo-de-tratamento-mensal-por-doente
acreditacao-de-unidades-de-saude
evolucao-mensal-distribuicao-novo-coronavirus-2019-n-cov-para-extracao-de-rna-e-
taxa-de-mortalidade-por-avc-isquemico-e-hemorragico
ausencias-para-formacao-e-aperfeicoamento-profissional
acesso-de-consultas-medicas-pela-populacao-inscrita
orgaos-colhidos-e-transplantados
monitorizacao-sazonal-csp
agregados-economico-financeiros
antibioticos
cartao-da-pessoa-com-doenca-rara
reservas
satisfacao-do-utente-em-atendimento-telefonico
demora-media-antes-da-cirurgia
chamadas-capic
despesa-com-medicamentos-no-ambulatorio-sns
antibioticos-carbapenemes
evolucao-mensal-do-no-de-chamadas-atendidas-no-centro-de-informacao-antivenenos
dadores-vivos-e-falecidos
trabalhadores-por-modalidade-de-vinculacao
no-de-entidades-inspe

In [4]:

RECORDS_PAGE_LIMIT = 100      # typical /records cap on Opendatasoft v2.1
RECORDS_MAX_WINDOW = 10000    # many domains enforce offset+limit <= 10k on /records

def _get_json(url, params=None, timeout=120):
    r = requests.get(url, params=params or {}, timeout=timeout)
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        try:
            print("Server error payload:", r.json())
        except Exception:
            print("Server error text:", r.text[:1000])
        raise e
    return r.json()

def _fetch_via_records(dataset_id: str) -> pd.DataFrame:
    """
    Pull up to ~10k rows via JSON paging (no select/where/order_by).
    Use exports for full extracts beyond this window.
    """
    url = f"{BASE}/catalog/datasets/{dataset_id}/records"
    offset, out = 0, []
    while True:
        if offset >= RECORDS_MAX_WINDOW:
            break
        limit = min(RECORDS_PAGE_LIMIT, RECORDS_MAX_WINDOW - offset)
        payload = _get_json(url, params={"limit": limit, "offset": offset}, timeout=120)
        rows = payload.get("results") or payload.get("records") or []
        if not rows:
            break
        for row in rows:
            out.append(row["record"] if isinstance(row, dict) and "record" in row else row)
        offset += limit
    return pd.json_normalize(out)

def _fetch_via_export_csv_robust(dataset_id: str) -> pd.DataFrame:
    """
    Robust CSV loader for Opendatasoft exports:
      - handles BOM (utf-8-sig)
      - auto-detects delimiter (comma/semicolon/tab/pipe) with csv.Sniffer
      - if sep is known: try fast C engine first
      - if needed: fall back to engine='python' (no low_memory), on_bad_lines='skip'
    """
    base_csv = f"{BASE}/catalog/datasets/{dataset_id}/exports/csv?use_labels_for_header=false"

    # 1) Probe a small sample to detect delimiter
    head_bytes = requests.get(base_csv, timeout=120).content[:100_000]
    sample = head_bytes.decode("utf-8-sig", errors="replace")
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=[",",";","\t","|"])
        sep = dialect.delimiter
    except Exception:
        sep = None

    # 2) Try with C engine if we know the separator (faster, tolerant in most cases)
    try:
        if sep is not None:
            return pd.read_csv(base_csv, sep=sep, encoding="utf-8-sig")
    except Exception as e:
        print(f"[CSV C-engine attempt failed: {e}]")

    # 3) Fall back to Python engine (no low_memory here), allow skipping malformed rows
    try:
        return pd.read_csv(
            base_csv,
            sep=sep, engine="python", encoding="utf-8-sig",
            on_bad_lines="skip"
        )
    except Exception as e:
        print(f"[CSV python-engine streaming failed: {e}]")
        # 4) Last resort: download to temp then read (sometimes helps behind proxies)
        with requests.get(base_csv, stream=True, timeout=300) as r:
            r.raise_for_status()
            with tempfile.NamedTemporaryFile(delete=False, suffix=".csv") as tmp:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    tmp.write(chunk)
                tmp_path = tmp.name
        try:
            # Try C engine first if sep is known, else Python engine
            if sep is not None:
                return pd.read_csv(tmp_path, sep=sep, encoding="utf-8-sig")
            else:
                return pd.read_csv(tmp_path, sep=None, engine="python", encoding="utf-8-sig", on_bad_lines="skip")
        finally:
            try: os.remove(tmp_path)
            except Exception: pass

def fetch_full_dataset(dataset_id: str, prefer: str = "auto") -> pd.DataFrame:
    """
    Fetch ALL rows with no filters/order.
    prefer='auto': try /records (<= ~10k), else CSV export robust reader.
    prefer='records': force records-only (<= ~10k).
    prefer='export': force CSV export robust reader.
    """
    if prefer == "records":
        return _fetch_via_records(dataset_id)
    if prefer == "export":
        return _fetch_via_export_csv_robust(dataset_id)

    # auto: try records first; if likely truncated or fails, use export
    try:
        df_rec = _fetch_via_records(dataset_id)
        if len(df_rec) >= RECORDS_MAX_WINDOW:
            return _fetch_via_export_csv_robust(dataset_id)
        return df_rec
    except requests.HTTPError:
        return _fetch_via_export_csv_robust(dataset_id)

DATASET_ID = "despesa-com-medicamentos-nos-hospitais-do-sns"

# Auto → records first (fast), fallback to CSV export (full), no pyarrow needed
df = fetch_full_dataset(DATASET_ID, prefer="auto")

print(f"Total rows fetched: {len(df)}")
display(df)

# Optional: show ALL rows/cols in output (can be heavy!)
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", 2000)
# pd.set_option("display.max_colwidth", None)
# display(df)

Total rows fetched: 840


,tempo,regiao,encargos_sns_hospitalar
0,2011-01,Algarve,3480241.41
1,2011-02,Lisboa e Vale do Tejo,38689777.33
2,2013-01,Lisboa e Vale do Tejo,41423750.34
3,2013-07,Alentejo,2849299.56
4,2014-03,Lisboa e Vale do Tejo,36887664.83
...,...,...,...
835,2021-02,Norte,40506922.62
836,2021-09,Alentejo,3863549.94
837,2022-02,Norte,49156193.45
838,2022-04,Lisboa e Vale do Tejo,62488121.86


In [33]:
# ---------- CSV export loader (robust) ----------
def _read_od_csv(dataset_id: str) -> pd.DataFrame:
    url = f"{BASE}/catalog/datasets/{dataset_id}/exports/csv?use_labels_for_header=false"
    head = requests.get(url, timeout=120).content[:100_000]
    sample = head.decode("utf-8-sig", errors="replace")
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=[",",";","\t","|"])
        sep = dialect.delimiter
    except Exception:
        sep = None
    try:
        if sep is not None:
            return pd.read_csv(url, sep=sep, encoding="utf-8-sig")
        else:
            return pd.read_csv(url, sep=None, engine="python", encoding="utf-8-sig")
    except Exception:
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with tempfile.NamedTemporaryFile(delete=False, suffix=".csv") as tmp:
                for chunk in r.iter_content(chunk_size=1<<20):
                    tmp.write(chunk)
                tmp_path = tmp.name
        try:
            if sep is not None:
                return pd.read_csv(tmp_path, sep=sep, encoding="utf-8-sig")
            else:
                return pd.read_csv(tmp_path, sep=None, engine="python", encoding="utf-8-sig")
        finally:
            try: os.remove(tmp_path)
            except Exception: pass

# ---------- helpers ----------
def _first_series(df: pd.DataFrame, colname: str):
    if colname not in df.columns: return None
    idxs = [i for i, c in enumerate(df.columns) if c == colname]
    return df.iloc[:, idxs[0]] if idxs else None

def _override_known_cols(dataset_id: str, df: pd.DataFrame):
    # Exact ids used by SNS medication datasets
    if dataset_id == "despesa-com-medicamentos-nos-hospitais-do-sns":
        date_col = "tempo" if "tempo" in df.columns else None
        region_col = "regiao" if "regiao" in df.columns else None
        value_col = "encargos_sns_hospitalar" if "encargos_sns_hospitalar" in df.columns else None
        return date_col, region_col, value_col
    if dataset_id == "despesa-com-medicamentos-no-ambulatorio-sns":
        date_col = "tempo" if "tempo" in df.columns else None
        region_col = "regiao" if "regiao" in df.columns else None
        candidates = [c for c in df.columns if c.lower().startswith("encargos_sns_ambulatorio")]
        value_col = candidates[0] if candidates else ("encargos_sns_ambulatorio" if "encargos_sns_ambulatorio" in df.columns else None)
        return date_col, region_col, value_col
    return None, None, None

def _detect_date_col(df: pd.DataFrame):
    name_candidates = [c for c in df.columns if any(k in c.lower() for k in ["tempo","data","date","dt","timestamp","periodo","period"])]
    for c in name_candidates:
        s = pd.to_datetime(_first_series(df, c), errors="coerce")
        if s.notna().any():
            return c
    year = next((c for c in df.columns if re.fullmatch(r"(?i)ano|year", c) or re.search(r"(?i)\bano\b|\byear\b", c or "")), None)
    month = next((c for c in df.columns if re.fullmatch(r"(?i)mes|month", c) or re.search(r"(?i)\bmes\b|\bmonth\b", c or "")), None)
    if year and month: return (year, month)
    return None

def _detect_region_col(df: pd.DataFrame):
    for c in df.columns:
        if any(k in c.lower() for k in ["regiao","região","ars","acronimo_ars","regiao_saude","regiao_administrativa"]):
            return c
    return None

def _detect_hospital_col(df: pd.DataFrame):
    for c in df.columns:
        if any(k in c.lower() for k in ["hospital","entidade","instituicao","unidade","uls","ch","ipo","centro hospitalar"]):
            return c
    return None

def _detect_value_col(df: pd.DataFrame):
    prefer = ["encargos","despesa","gasto","custo","valor","montante"]
    numeric_cols = []
    for c in df.columns:
        s = _first_series(df, c)
        if s is not None and pd.api.types.is_numeric_dtype(s):
            numeric_cols.append(c)
    best, best_score, best_n = None, -1, -1
    for c in numeric_cols:
        score = sum(k in c.lower() for k in prefer)
        n = _first_series(df, c).notna().sum()
        if score > best_score or (score == best_score and n > best_n):
            best, best_score, best_n = c, score, n
    if best is None:
        for c in df.columns:
            if any(k in c.lower() for k in prefer):
                s = _first_series(df, c)
                if s is None: continue
                if s.dtype == 'object':
                    s2 = pd.to_numeric(s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False), errors="coerce")
                else:
                    s2 = pd.to_numeric(s, errors="coerce")
                if s2.notna().any():
                    df[c] = s2
                    return c
    return best

# ---------- main (with Plotly hover) ----------
def plot_medication_spend_evolution(
    region: str = None,
    hospital: str = None,
    start: str = None,
    end: str = None,
    sources=("hospital","ambulatory"),
    split_by_source: bool = False,
    figsize=(10,5),
    return_data=False,
    debug: bool = False,
    renderer: str = "plotly"   # "plotly" (interactive) or "matplotlib" (static)
):
    source_to_dataset = {
        "hospital":   "despesa-com-medicamentos-nos-hospitais-do-sns",
        "ambulatory": "despesa-com-medicamentos-no-ambulatorio-sns"
    }
    ds_ids = [source_to_dataset[s] for s in sources if s in source_to_dataset]

    frames = []
    for s, dsid in zip(sources, ds_ids):
        df = _read_od_csv(dsid)
        if debug: print(f"[{dsid}] loaded shape={df.shape}")

        # 1) columns
        date_col, region_col, value_col = _override_known_cols(dsid, df)
        if not date_col:
            dc = _detect_date_col(df)
            if isinstance(dc, tuple):
                y, m = dc
                yv = pd.to_numeric(_first_series(df, y), errors="coerce")
                mv = pd.to_numeric(_first_series(df, m), errors="coerce").fillna(1)
                dt = pd.to_datetime(dict(year=yv.astype("Int64"), month=mv.astype("Int64"), day=1), errors="coerce")
                df["__period__"] = dt.dt.to_period("M").dt.to_timestamp("M")
            elif dc:
                s_date = pd.to_datetime(_first_series(df, dc), errors="coerce")
                df["__period__"] = s_date.dt.to_period("M").dt.to_timestamp("M")
                date_col = dc
            else:
                df["__period__"] = pd.NaT
        else:
            s_date = pd.to_datetime(_first_series(df, date_col), errors="coerce")
            df["__period__"] = s_date.dt.to_period("M").dt.to_timestamp("M")

        if not region_col:
            region_col = _detect_region_col(df)
        hosp_col = _detect_hospital_col(df)
        if not value_col:
            value_col = _detect_value_col(df)
        if debug:
            print(f"  date_col={date_col}, region_col={region_col}, hosp_col={hosp_col}, value_col={value_col}")

        if not value_col:
            if debug: print(f"[WARN] No value column found in {dsid}; skipping.")
            continue

        # 2) filters
        if region and region_col in df.columns:
            sr = _first_series(df, region_col).astype(str).str.lower()
            df = df[sr == region.lower()]
        if hospital and hosp_col in df.columns:
            sh = _first_series(df, hosp_col).astype(str)
            df = df[sh.str.contains(hospital, case=False, na=False)]

        def _to_period_start(x):
            if x is None: return None
            if re.fullmatch(r"\d{4}-\d{2}", x): return pd.to_datetime(x + "-01")
            if re.fullmatch(r"\d{4}", x):       return pd.to_datetime(x + "-01-01")
            return None

        p_start = _to_period_start(start)
        p_end   = _to_period_start(end)
        if p_start is not None:
            df = df[df["__period__"] >= p_start]
        if p_end is not None:
            df = df[df["__period__"] <= pd.to_datetime(p_end).to_period("M").to_timestamp("M")]

        # 3) numeric value series
        s_val = _first_series(df, value_col)
        if s_val is None:
            if debug: print(f"[WARN] Value col {value_col} missing in {dsid}; skipping.")
            continue
        if s_val.dtype == 'object':
            s_val = pd.to_numeric(s_val.str.replace(".", "", regex=False).str.replace(",", ".", regex=False), errors="coerce")
        else:
            s_val = pd.to_numeric(s_val, errors="coerce")

        frames.append(pd.DataFrame({
            "__period__": df["__period__"],
            "__value__":  s_val,
            "__source__": s
        }))

    if not frames:
        raise ValueError("No data loaded. Enable debug=True to inspect detection and filters.")

    full = pd.concat(frames, ignore_index=True)
    full = full[full["__period__"].notna() & full["__value__"].notna()]
    if full.empty:
        raise ValueError("All rows were NaN after parsing/filters. Enable debug=True to inspect.")

    # aggregate
    if split_by_source:
        agg = full.groupby(["__period__","__source__"], as_index=False)["__value__"].sum()
    else:
        agg = full.groupby(["__period__"], as_index=False)["__value__"].sum()
        agg["__source__"] = "Total"

    agg = agg.sort_values("__period__")
    agg = agg.rename(columns={"__period__":"period","__value__":"value","__source__":"source"})
    agg["period"] = pd.to_datetime(agg["period"])  # ensure datetime for Plotly axis

   # ---------- render ----------
    title = "Evolução da despesa em medicamentos"
    bits = []
    if region:   bits.append(f"Região: {region}")
    if hospital: bits.append(f"Hospital contém: {hospital}")
    if start or end: bits.append(f"Período: {start or '...'} → {end or '...'}")
    if bits: title += "  —  " + " | ".join(bits)

    import plotly.express as px
    import plotly.io as pio

    if split_by_source:
        fig = px.line(
            agg, x="period", y="value", color="source",
            markers=True, title=title,
            hover_data={"period": "|%Y-%m", "value": ":,.2f", "source": True}
        )
        fig.update_traces(
            hovertemplate="<b>%{customdata[0]}</b><br>Período: %{x|%Y-%m}<br>Despesa: € %{y:,.2f}<extra></extra>",
            customdata=agg[["source"]],
        )
    else:
        fig = px.line(
            agg, x="period", y="value",
            markers=True, title=title,
            hover_data={"period": "|%Y-%m", "value": ":,.2f"}
        )
        fig.update_traces(
            hovertemplate="Período: %{x|%Y-%m}<br>Despesa: € %{y:,.2f}<extra></extra>"
        )

    fig.update_layout(
        xaxis_title="Período (mensal)",
        yaxis_title="Despesa (€)",
        hovermode="x unified"
    )

    # Show with the currently selected default renderer:
    fig.show()

    if return_data:
        return fig, agg
    return fig

    # Matplotlib fallback (static, no hover)
    plt.figure(figsize=figsize)
    if split_by_source:
        for src, sub in agg.groupby("source"):
            plt.plot(sub["period"], sub["value"], marker="o", label=src.capitalize())
        plt.legend(title="Fonte")
    else:
        plt.plot(agg["period"], agg["value"], marker="o", color="#1f77b4")
    plt.title(title)
    plt.xlabel("Período (mensal)")
    plt.ylabel("Despesa (€)")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()

    if return_data:
        return plt.gcf(), agg
    return plt.gcf()

In [34]:
# 1) Default (interactive Plotly with hover): both sources, no filters
plot_medication_spend_evolution()

# 2) Region and time window filters:
plot_medication_spend_evolution(region="Centro", start="2023", end="2025-12")

# 3) Comparison between the 2 types of spending (interactive hover with source split):
fig, data = plot_medication_spend_evolution(split_by_source=True, return_data=True)
display(data.head())

,period,source,value
0,2011-01-31,ambulatory,1.051290e+08
1,2011-01-31,hospital,8.890101e+07
2,2011-02-28,ambulatory,1.034359e+08
3,2011-02-28,hospital,8.446928e+07
4,2011-03-31,ambulatory,1.164680e+08
